# Decision Trees & Random forests

## Imports

## Load data

Weather data from Australia. The task is to predict if tomorrow is going to rain based on today's weather characteristics.

Source: https://www.kaggle.com/jsphyg/weather-dataset-rattle-package

**Target description**

- RainTomorrow: 1 if precipitation (mm) in the 24 hours to 9am exceeds 1mm, otherwise 0 for the next day

**Predictors description**

- MinTemp: The minimum temperature in degrees celsius
- MaxTemp: The maximum temperature in degrees celsius
- Sunshine: The number of hours of bright sunshine in the day.
- WindGustSpeed: The speed (km/h) of the strongest wind gust in the 24 hours to midnight
- Humidity9am: Humidity (percent) at 9am
- Humidity3pm: Humidity (percent) at 3pm
- Pressure9am: Atmospheric pressure (hpa) reduced to mean sea level at 9am
- Pressure3pm: Atmospheric pressure (hpa) reduced to mean sea level at 3pm
- RainToday: 1 if precipitation (mm) in the 24 hours to 9am exceeds 1mm, otherwise 0




In [ ]:
# Import dataset
data_file = Path("../Data/weatherAUS.csv")
data_all = pd.read_csv(data_file, encoding='utf-8')

# Keep only specified columns and rows
columns = ["Date", "MinTemp", "MaxTemp", "Sunshine", "WindGustSpeed", "Humidity9am", "Humidity3pm", "Pressure9am", "Pressure3pm", "RainToday", "RainTomorrow"]
data = data_all.loc[data_all['Date']>='2016-01-01', columns]

# Convert date to datetime format
data['Date'] = pd.to_datetime(data['Date'])

print(f'Number of rows:      {data.shape[0]}')
print(f'Number of columns:   {data.shape[1]}')

# CART

Implementation: https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html#sklearn.tree.DecisionTreeClassifier

We use implementation from sci-kit learn package. The implementation has several characteristics:

- Missing values are not supported
- Assumes the predictors to be numeric variables (either continuous or ordinal categories). 
- Both categorical and continuous targets are possible (we chose either DecisionTreeClassification or DecisionTreeRegression class according to our task)

Short description:

In each node we iterate over randomly chosen set of predictors and compute gain for each predictors measured using defined measure and assuming different possible splits over this predictor. Using the best predictor we perform split into two subtrees. See details of the implementation for further parameters that influence stopping of the algorithm.

We copy our data to 'data_cart' dataframe.

First, we handle missing values, then encode all predictors as numerical.

In [ ]:
data_cart.head()

In [ ]:
# We check the format of the data
data_cart.dtypes

In [ ]:
data_cart['RainTodayFlag'] = ...
data_cart['RainTomorrowFlag'] = ...

In [ ]:
# another way how to reencode using faster np.where function
all(np.where(data_cart['RainToday']=='Yes', 1, 0) == data_cart['RainToday'].apply(lambda x: 1 if x=='Yes' else 0))

In [ ]:
predictors

In [ ]:
# Number of missing values in each column
data_cart.isna().sum()

In [ ]:
data_cart[predictors + [target]].isna().sum()

## Split & train

In [ ]:
# we have observation from 2 years
data_cart['Month'] = data_cart['Date'].dt.strftime('%m')
data_cart['Month'].value_counts()

In [ ]:
data_cart['YearMon'] = data_cart['Year'].astype(str) + (data_cart['Month']).astype(str)
data_cart['YearMon'].value_counts()

In [ ]:
data_cart_train, data_cart_test = ...

In [ ]:
# Now we can fit the model.

criterion = ... #(TODO)
splitter = ... #(TODO)
mdepth = ... #(TODO)
min_samples_split = ... #(TODO)
min_samples_leaf = ... #(TODO)
random_state = 17

# Now we contruct the model.
model = tree.DecisionTreeClassifier(
    criterion=criterion,
    splitter=splitter, 
    max_depth=mdepth,
    min_samples_split=min_samples_split,
    min_samples_leaf=min_samples_leaf, 
    random_state=random_state, 
)

# And fit training data.

clf = model.fit(data_cart_train[predictors], data_cart_train[target])

# Predict class labels on training data
pred_labels_tr = model.predict(data_cart_train[predictors])
# Predict class labels on a test data
pred_labels_te = model.predict(data_cart_test[predictors])

# Tree summary and model evaluation metrics
print('*************** Tree Summary ***************')
print('Classes: ', clf.classes_)
print('Tree Depth: ', clf.tree_.max_depth)
print('No. of leaves: ', clf.tree_.n_leaves)
print('No. of features: ', clf.n_features_in_)
print('--------------------------------------------------------')
print("")

print('*************** Evaluation on Test Data ***************')
score_te = model.score(data_cart_test[predictors], data_cart_test[target])
print('Accuracy Score: ', score_te)
# Look at classification report to evaluate the model
print(classification_report(data_cart_test[target], pred_labels_te))
print('--------------------------------------------------------')
print("")

print('*************** Evaluation on Training Data ***************')
score_tr = model.score(data_cart_train[predictors], data_cart_train[target])
print('Accuracy Score: ', score_tr)
# Look at classification report to evaluate the model
print(classification_report(data_cart_train[target], pred_labels_tr))
print('--------------------------------------------------------')

## Evaluate

### Accuracy

In [ ]:
data_cart_test['Prediction_prob'] = ...
data_cart_test['Prediction'] = ...

In [ ]:
...

### Predictor importance

In scikit-learn the feature importance is the decrease in node impurity. The key is that it measures the importance only at a node level. Then, all the nodes are weighted by how many samples reach that node.

In [ ]:
# Plot importance
x_values = list(range(len(importances)))
# Make a bar chart
plt.bar(x_values, importances, orientation = 'vertical')
# Tick labels for x axis
plt.xticks(x_values, predictors, rotation='vertical')
# Axis labels and title
plt.ylabel('Importance'); plt.xlabel('Variable'); plt.title('Variable Importances');

## Data Manipulation

In [ ]:
data_chaid = data.copy()

In [ ]:
data_chaid=data_chaid[pd.isnull(data_chaid['RainTomorrow'])==False] 
data_chaid=data_chaid # no need to fill missing predictors in this implementation

In [ ]:
data_chaid.dtypes

In [ ]:
target = 'RainTomorrowFlag'

predictors_chaid = list(data_chaid.columns)
predictors_chaid.remove('Date') # Date will not be used for training
predictors_chaid.remove('RainToday') # We use RainTodayFlag instead
predictors_chaid.remove('RainTomorrow') # Target
predictors_chaid.remove('RainTomorrowFlag') # Target# We denote 'target' the name of the column with target values. 'predictors' stands for the list of all predictors

In [ ]:
# We check the predictors if we can assume that these are continuous

for predictor in predictors_chaid:
    
    plt.hist(data_chaid[predictor], density=True, bins=10)  # density=False would make counts
    plt.ylabel('Probability')
    plt.xlabel(predictor);

    plt.show()

Based on the histograms above we conclude that some of the predictors must be categorized. We do the categorization based on training data only, therefore, now we have to split the dataset.

In [ ]:
# Only 'RainTodayFlag' predictor is categorical. The rest is continuous, therefore we have to categorize.

n_cat = 10 # number of categories created for each predictor

# We store categorized predictors from 'predictors_chaid' to 'predictors_chaid_cat' columns.

predictors_chaid_cat = []
for predictor in predictors_chaid:
    
    predictor_cat = predictor+'_cat'
    predictors_chaid_cat.append(predictor_cat)
    
    if (predictor not in ['RainTodayFlag']): # 'RainTodayFlag' need not to be categorized
        
        # TODO: Categorize given predictor 'predictor' by quantiles using function pd.qcut with 'n_cat' categories. 
        # Note that in the future we will need to categorize also test data, therefore, label the
        # new categories by right-end-point of each interval (np.infty for infinity).
        data_chaid_train[predictor_cat] = pd.qcut(data_chaid_train[predictor], q=n_cat, labels=None, duplicates='drop') #(TODO)
        data_chaid_train[predictor_cat] = data_chaid_train[predictor_cat].apply(lambda x: x.right).astype(float) #(TODO)
        pred_max = data_chaid_train[predictor_cat].max() #(TODO)
        data_chaid_train.loc[data_chaid_train[predictor_cat] == pred_max, predictor_cat] = np.infty #(TODO)
        
    else:
        # Already categorized
        data_chaid_train[predictor_cat] = data_chaid_train[predictor]


In [ ]:
# Now we can fit the model.

# The model needs to know if 'ordinal' or 'nominal' predictors are present (only 'nominal' in our case as we have also missing values)
independent_columns = dict(zip(predictors_chaid_cat, ['nominal'] * len(predictors_chaid_cat)))

# Choose the parameters carefully. See the algorithm description above.
config_chaid = {
    'alpha_merge': 0.05, # minimum significance level for categories not to be merged 
    'max_depth': 3, #maximum depth of tree 
    'min_parent_node_size': 30, # minimum number of observations of node for splitting 
    'min_child_node_size': 15, # minimum number of observations in new node 
    'split_threshold': 0, # minimum gain for new split
}

tree_ch = CHAID.Tree.from_pandas_df(
    data_chaid_train,
    independent_columns,
    target,
    alpha_merge = config_chaid['alpha_merge'],
    max_depth = config_chaid['max_depth'],
    min_parent_node_size = config_chaid['min_parent_node_size'],
    min_child_node_size = config_chaid['min_child_node_size'],
    split_threshold = config_chaid['split_threshold'],
)

# Tree summary and model evaluation metrics
print('*************** Tree Summary ***************')
print('Classes: ', len(data_chaid_train[target].unique()))
print('Tree Depth: ', tree_ch.max_depth)
print('No. of features: ', len(independent_columns))
print('--------------------------------------------------------')
print("")

print('*************** Evaluation on Training Data ***************')
print('Accuracy Score: ', tree_ch.accuracy())

## Evaluate

### Predict

In [ ]:
# Disclaimer: The implementation does not involve method for prediction. Therefore, we use our own implementation.

# Leaves of the tree: list of leaves with rules
leaves = tree_ch.classification_rules() # Contains rules for leaves
nodes = tree_ch.tree_store # Contains predictions for all nodes (not only leaves)

def chaid_predict(data):
    """
        Returns prediction of 'target' using trained CHAID tree for one observation by identifying correct leaf.
        
        Args:
            one observation as in format 'data_chaid' with columns 'predictors_chaid_cat'
        
        Returns:
            float: number of positive cases in identified leaf/number of total cases in identified leaf
    """

    # Iterate over all leaves of the tree and find the correct one
    for leaf in leaves:
        
        # List of rules that determine the leaf
        rules = leaf['rules']
        rule_i = 0

        # Evaluate conditions until one fails
        is_member = True
        while is_member and (rule_i+1<=len(rules)):
            
            
            var_name = rules[rule_i]['variable']
            var_val_list = rules[rule_i]['data']
            
            if data[var_name] not in var_val_list:
                is_member = False
            
            rule_i = rule_i+1
            
        if is_member: # All conditions met
            
            zeros = nodes[leaf['node']]._members[0]
            ones = nodes[leaf['node']]._members[1]
            
            return ones/(zeros+ones)
        
    return None # No leaf found

### Accuracy

In [ ]:
# Inspect stability of the model performance in time

ax1 = plt.subplot(111)
ax1.bar(range(len(data_chaid_accuracy)), data_chaid_accuracy)
ax1.set_xticks(range(len(data_chaid_accuracy)))
ax1.set_xticklabels(data_chaid_accuracy.index, rotation=90)
ax1.set_xlabel('Month')
ax1.set_ylabel('Accuracy')

plt.show()

In [ ]:
import random

random.seed(17)

In [ ]:
# Meta parameters
target = 'RainTomorrowFlag'

n_models = 10 # Number of trees in ensemble
p_obs = 0.3 # Prob. that observation is chosen for training of a given tree

criterion = ... 
splitter = ... 
mdepth = ... 
min_samples_split = ...
min_samples_leaf = ... 


# We fit 'n_models' models and store them into list
clf_list = []
for n in range(n_models):
    
    # TODO: Define 'train_index' a list of indices from 'data_cart_train' to be used for training in given iteration, consider parameter 'p_obs'
    train_index = ...

    # Init the model
    model = tree.DecisionTreeClassifier(
        criterion=criterion,
        splitter=splitter, 
        max_depth=mdepth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf, 
        random_state=n,
    )

    clf = model.fit(data_cart_train.loc[train_index, predictors], data_cart_train.loc[train_index, target])
    
    clf_list = clf_list + [model]

    if n==0:
        # Tree summary and model evaluation metrics
        print('*************** Tree Summary ***************')
        print('Classes: ', clf.classes_)
        print('Tree Depth: ', clf.tree_.max_depth)
        print('No. of leaves: ', clf.tree_.n_leaves)
        print('No. of features: ', clf.n_features_in_)
        print('--------------------------------------------------------')
        print("")
        
        # TODO: Compute predictions using model.predict_proba for 'data_cart_train' and 'data_cart_test' in the first iteration
        data_cart_train['Prediction_forest_prob'] = ...
        data_cart_test['Prediction_forest_prob'] = ...
    
    else:
        # TODO: Compute predictions using model.predict_proba for 'data_cart_train' and 'data_cart_test' after the first iteration
        data_cart_train['Prediction_forest_prob'] = ...
        data_cart_test['Prediction_forest_prob'] = ...
    
    # TODO: Use threshold to transform probabilities to 0/1 values and store prediction as 'Prediction_forest' column in 'data_cart_train' and 'data_cart_test' datasets
    data_cart_train['Prediction_forest'] = ...
    data_cart_test['Prediction_forest'] = ...

    print('*************** Evaluation After Iteration '+str(n+1)+' *********')
    score_te = 1-abs(data_cart_train['Prediction_forest']-data_cart_train[target]).mean()
    print('Accuracy Train: ', score_te)
    score_tr = 1-abs(data_cart_test['Prediction_forest']-data_cart_test[target]).mean()
    print('Accuracy Valid: ', score_tr)
    print('--------------------------------------------------------')


# Bonus task

Use different approach for training our random forest: Choose randomly predictors which will be used for training each of the tree.

For this task we use different data points from the same dataset as above. Therefore, we reload the data.

In [ ]:
# Import dataset
data_file = Path("../Data/weatherAUS.csv")
data = pd.read_csv(data_file, encoding='utf-8')

# Keep only specified columns and rows
columns = ["Rainfall", "Evaporation", "Sunshine", "Location", "WindGustSpeed", "WindDir9am", "WindDir3pm", "Humidity9am", "Humidity3pm",  "Pressure9am", "Pressure3pm", "Cloud9am", "Cloud3pm", "MaxTemp"]
data_forest = data.loc[data['Location'].isin(["Canberra", "Sydney", "Albury"]), columns]

print(f'Number of rows:      {data_forest.shape[0]}')
print(f'Number of columns:   {data_forest.shape[1]}')

In [ ]:
# Descriptive statistics: note that only numerical features are included
data_forest.describe()

In [ ]:
# Number of missing values in each column
data_forest.isna().sum()

In [ ]:
# We denote 'target' the name of the column with target values. 
# 'predictors_cat' stands for the list categorical predictors
# 'predictors_num' stands for the list numerical predictors

target = "MaxTemp"

predictors_cat = list(data_forest.columns[[data_forest[col].dtype == 'object' for col in data_forest.columns]])
predictors_num = list(data_forest.columns[[data_forest[col].dtype != 'object' for col in data_forest.columns]])
predictors_num.remove(target)

### Numerical features

In [ ]:
# Split the data. 
X_train, X_test = train_test_split(data_forest, test_size=0.2, random_state=17)

data_forest.loc[X_train.index, 'sample'] = 'train'
data_forest.loc[X_test.index, 'sample'] = 'test'

# define sample masks
train_mask = (data_forest['sample'] == 'train')
test_mask = (data_forest['sample'] == 'test')

<span style="color:red">**TO DO:** </span> \
Use pd.get_dummies to encode categorical predictors. Store the result to 'data_one_hot' dataframe

Now, we create function that computes encoding values for given predictor.

For binary target, the encoding for given predictor and category is given as a convex combination of mean target value for observations in the category and mean target value for the whole dataset:

\begin{align*}
\frac{\frac{count_{cat}}{count_{all}}*mean\_target_{cat}+\alpha*mean\_target_{all}}{\frac{count_{cat}}{count_{all}}+\alpha},
\end{align*}
where $\alpha$ is a chosen parameter.

In [ ]:
# Apply encoding. Names of columns of new columns will be stored in 'predictors_mean_target'

predictors_mean_target = []
encodings = {}
for pred in predictors_cat:
    
    # New predictor name contains 'MT' and former predictor name
    pred_mean_target = pred + "MT"
    
    # First, compute the encoding for given predictor. Then apply using 'map' method
    # that works on data frames and store values to 'data_forest[pred_mean_target]'
    
    encodings[pred_mean_target] = mean_target_encoding(data_forest[train_mask], pred, target) 
    data_forest[pred_mean_target] = data_forest[pred].map(encodings[pred_mean_target]).fillna(sum(encodings[pred_mean_target].values()) / len(encodings[pred_mean_target]))
    
    # Keep new predictors names
    predictors_mean_target = predictors_mean_target + [pred_mean_target]

## Fit

We train separately based on different encodings.

### One-hot encoding

In [ ]:
...

<span style="color:red">**TO DO:** </span> \
Use [sklearn RandomForestRegressor documentation](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html) to carefully choose parameters, init and fit the model. Use 'X_train_oh' and 'y_train_oh' data defined above.

## Evaluate

We evaluate both models based on MAE, MAPE and visually by inspecting the prediction errors. We also consider features' importance.

In scikit-learn the feature importance is the decrease in node impurity. The key is that it measures the importance only at a node level. Then, all the nodes are weighted by how many samples reach that node.

In [ ]:
# Use the forest's predict method on the test data
predictions_oh = rf_oh.predict(X_test_oh)

# Calculate MAE
absolute_error_oh = abs(predictions_oh - y_test_oh)
MAE_oh = round(np.mean(absolute_error_oh), 2)

# Calculate MAPE
absolute_percentage_error_oh = 100 * (absolute_error_oh / y_test_oh)
MAPE_oh = round(np.mean(absolute_percentage_error_oh), 2)

# Print
print('Mean Absolute Error:', MAE_oh, 'degrees.')
print('Mean Absolute Percentage Error:', MAPE_oh, '%.')

### Mean target encoding

In [ ]:
# Get numerical feature importances
importances_mt = list(rf_mt.feature_importances_)

# List of tuples with variable and importance
feature_importances_mt = [(feature, round(importance, 2)) for feature, importance in zip(predictors_num+predictors_mean_target, importances_mt)]

# Sort the feature importances by most important first and take only 10 most powerful
feature_importances_mt = sorted(feature_importances_mt, key = lambda x: x[1], reverse = True)[:10]

# Print out the feature and importances 
[print('Variable: {:20} Importance: {}'.format(*pair)) for pair in feature_importances_mt];

In [ ]:
# Visualise prediction errors. Namely, we plot difference in residuals of the two models against target value in scatter plot.

fig, ax1 = plt.subplots(figsize=(10, 10))
ax1.scatter(y_test_mt, predictions_mt-predictions_oh)
ax1.set_xlabel('Target', size=20)
ax1.set_ylabel('Errors difference (mean-target - one-hot)', size=20)

plt.show()

## Reencoding for CART

## Data split

## Fit the model

## Prediction

## Accuracy

## Random forest manually

In [ ]:
        # # TODO: Compute predictions using model.predict_proba for 'data_cart_train' and 'data_cart_test' in the first iteration
        # data_cart_train['Prediction_forest_prob'] = model.predict_proba(data_cart_train[predictors])[:,1] 
        # data_cart_test['Prediction_forest_prob'] = model.predict_proba(data_cart_test[predictors])[:,1] 

In [ ]:
    # # TODO: Use threshold to transform probabilities to 0/1 values and store prediction as 'Prediction_forest' column in 'data_cart_train' and 'data_cart_test' datasets
    # data_cart_train['Prediction_forest'] = (data_cart_train['Prediction_forest_prob']>=0.5).astype(int)
    # data_cart_test['Prediction_forest'] = (data_cart_test['Prediction_forest_prob']>=0.5).astype(int)

In [ ]:
    # data_one_hot = pd.get_dummies(data_forest[pred], prefix=pred)

In [ ]:
# # Instantiate and fit model
# rf_oh = RandomForestRegressor(
#     n_estimators=500,
#     criterion='squared_error',
#     max_depth=7,
#     min_samples_leaf=10,
#     min_impurity_decrease=0.1,
#     bootstrap=True,
#     oob_score=False,
#     verbose=1,
# )
# rf_oh.fit(X_train_oh, y_train_oh)

In [ ]:
# # Instantiate and fit model
# rf_mt = RandomForestRegressor(
#     n_estimators=500,
#     criterion='squared_error',
#     max_depth=7,
#     min_samples_leaf=10,
#     min_impurity_decrease=0.1,
#     bootstrap=True,
#     oob_score=False,
#     verbose=1,
# )
# rf_mt.fit(X_train_mt, y_train_mt)